# **<center><span style= "color:#2F539B;">CNN with pyTorch</span></center>**

## ***<span style= "color:purple;">Imports </span>***

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

## ***<span style= "color:orange;">Dataset </span>***

In [2]:
import pandas as pd
df = pd.read_csv("datasets/fashion-mnist_train.csv")

In [3]:
torch.manual_seed(42)

In [4]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [6]:
X_train = X_train/255.0
X_test_test = X_test/255.0

### ***<span style= "color:#85BB65;"> creating custom dataset class </span>***

In [7]:
class CustomDataset(Dataset):
    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [8]:
train_dataset = CustomDataset(X_train, y_train)
len(train_dataset)

48000

In [9]:
test_dataset = CustomDataset(X_test, y_test)

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32, shuffle=False)

### ***<span style= "color:#85BB65;">Convolutional Neural Network class </span>***

In [11]:
class MyNN(nn.Module):
    
    def __init__(self, input_features):

        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),

        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 10)

        )
    
    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [12]:
epochs = 10
learning_rate = 0.1


In [13]:
model = MyNN(1)

#loss function
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

## ***<span style= "color:orange;">Model </span>***

In [14]:
for epoch in range(epochs):

    total_epoch_loss = 0
    
    for batch_features, batch_labels in train_loader:

        #forwardpass
        outputs = model(batch_features)

        #calcuate loss
        loss = criterion(outputs, batch_labels)

        # backpass
        optimizer.zero_grad()
        loss.backward()

        #update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()
    
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {avg_loss}')

Epoch: 1, Loss: 0.6014137823383013
Epoch: 2, Loss: 0.42027752271791297
Epoch: 3, Loss: 0.3584251050228874
Epoch: 4, Loss: 0.3139298191691438
Epoch: 5, Loss: 0.2918355849633614
Epoch: 6, Loss: 0.2664143724354605
Epoch: 7, Loss: 0.24492730089028678
Epoch: 8, Loss: 0.23318562366502982
Epoch: 9, Loss: 0.22020026241304974
Epoch: 10, Loss: 0.20928988222405315


### ***<span style= "color:#85BB65;">Model evaluation </span>***


In [23]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [24]:
# useful variables
total = 0
correct = 0

with torch.no_grad():
    
    for batch_features, batch_labels in test_loader:

        outputs = model(batch_features)

        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()
        

In [25]:
#acurracy
print(correct/total)

0.4811666666666667
